# UCF101 SSL implementation

This notebook adds an R3D-18 contrastive self-supervised experiment to the existing MSVD-QA pilot. Run the cells in order. First do the 32-video smoke run; only start the full run after it finishes successfully. UCF101 class labels are not used.

In [ ]:
from pathlib import Path
import argparse, sys, json, torch, torchvision

# Change these three paths to match your computer. CODE is the folder beside train.py.
CODE = Path(r'C:\Users\ASUS\VideoQA_Thesis\code')
DATA = CODE / 'data'
UCF_VIDEOS = Path(r'C:\PASTE\YOUR\UCF101\FOLDER\HERE')

assert (CODE / 'train.py').exists(), f'Cannot find train.py in {CODE}'
assert (UCF_VIDEOS.exists()), f'Change UCF_VIDEOS to the folder containing UCF101 videos: {UCF_VIDEOS}'
sys.path.insert(0, str(CODE))
print('PyTorch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
print('UCF101 videos:', sum(1 for p in UCF_VIDEOS.rglob('*') if p.suffix.lower() in {'.avi','.mp4','.mov','.mkv','.webm'}))
if not torch.cuda.is_available(): print('Warning: CPU will be very slow; use your CUDA-enabled Jupyter kernel.')

## 1. Quick smoke run (32 videos)

This is only a software/GPU check, not a thesis result. It should finish much faster than full training and create checkpoints under `runs/ucf101_r3d18_ssl_smoke`.

In [ ]:
from ssl_pretrain import main as pretrain_main

smoke_args = argparse.Namespace(
    videos=str(UCF_VIDEOS), output=str(CODE / 'runs' / 'ucf101_r3d18_ssl_smoke'),
    epochs=1, batch_size=2, workers=0, max_videos=32, val_fraction=0.125,
    temperature=0.07, lr=0.0001, weight_decay=0.01, seed=42,
    from_scratch=False, no_amp=False, resume=None
)
pretrain_main(smoke_args)
assert (CODE / 'runs' / 'ucf101_r3d18_ssl_smoke' / 'best.pt').exists()
print('Smoke run complete. Continue only if best.pt was created.')

## 2. Full UCF101 pretraining

This processes the full indexed video folder. On a 4 GB RTX 2050 it can take many hours. The live epoch/batch progress appears in this notebook cell. Checkpoints are saved after every epoch. If it is interrupted, use the resume cell below instead of restarting.

In [ ]:
full_ssl_dir = CODE / 'runs' / 'ucf101_r3d18_ssl_full'
full_args = argparse.Namespace(
    videos=str(UCF_VIDEOS), output=str(full_ssl_dir), epochs=10, batch_size=2, workers=0,
    max_videos=0, val_fraction=0.05, temperature=0.07, lr=0.0001, weight_decay=0.01,
    seed=42, from_scratch=False, no_amp=False, resume=None
)
pretrain_main(full_args)
print('Use this checkpoint for downstream features:', full_ssl_dir / 'best.pt')

### Resume after an interruption (run only if needed)

Set the target epoch count in `epochs`. For example, if the first run stopped at epoch 3, set `epochs=10` to continue through epoch 10.

In [ ]:
resume_args = argparse.Namespace(
    videos=str(UCF_VIDEOS), output=str(full_ssl_dir), epochs=10, batch_size=2, workers=0,
    max_videos=0, val_fraction=0.05, temperature=0.07, lr=0.0001, weight_decay=0.01,
    seed=42, from_scratch=False, no_amp=False, resume=str(full_ssl_dir / 'last.pt')
)
# Run this cell only when continuing an interrupted full run.
# pretrain_main(resume_args)

## 3. Cache MSVD-QA video features for the controlled comparison

We compare the same R3D-18 architecture with Kinetics-400 initialization versus additional UCF101 InfoNCE training. Separate caches prevent mixing representations. This step uses train and validation only.

In [ ]:
from r3d_features import main as r3d_features_main

CACHE_R3D_KINETICS = CODE / 'cache_r3d_kinetics'
CACHE_R3D_UCF_SSL = CODE / 'cache_r3d_ucf_ssl'
def make_r3d_cache(manifest, cache, checkpoint=None):
    args = argparse.Namespace(manifest=str(manifest), cache=str(cache), checkpoint=checkpoint,
        text='distilbert/distilbert-base-uncased', num_clips=8, clip_frames=8,
        clip_batch_size=2, text_batch_size=32, max_length=32)
    r3d_features_main(args)

for split in ['train', 'val']:
    make_r3d_cache(DATA / f'{split}.jsonl', CACHE_R3D_KINETICS)
    make_r3d_cache(DATA / f'{split}.jsonl', CACHE_R3D_UCF_SSL, str(full_ssl_dir / 'best.pt'))
print('Both train/validation caches are ready.')

## 4. Train matched QA models (validation only)

Each seed starts from the same random QA-head initialization in both conditions and gets the same 3,000 updates. Do not inspect test results or change settings based on them during this comparison.

In [ ]:
from train import main as train_main

def train_r3d_arm(cache, arm, seed):
    out = CODE / 'runs' / f'{arm}_s{seed}'
    args = argparse.Namespace(train=str(DATA / 'train.jsonl'), val=str(DATA / 'val.jsonl'),
        cache=str(cache), vocab=str(DATA / 'vocab.json'), output=str(out), steps=3000,
        batch_size=64, hidden=256, lr=0.0003, seed=seed, question_only=False,
        init=None, resume=None, eval_checkpoint=None, mode='full')
    print(f'\nTraining {arm}, seed {seed}', flush=True)
    train_main(args)

for seed in [42, 43, 44]:
    train_r3d_arm(CACHE_R3D_KINETICS, 'r3d_kinetics', seed)
    train_r3d_arm(CACHE_R3D_UCF_SSL, 'r3d_ucf_ssl', seed)

## 5. Final test (only after locking the protocol)

Run this cell only after choosing the model settings from validation and recording them. It creates test features and evaluates each saved seed once. Leave the gate `False` until the protocol is written down and frozen.

In [ ]:
FINAL_SSL_PROTOCOL_FROZEN = False
if not FINAL_SSL_PROTOCOL_FROZEN:
    raise RuntimeError('Freeze and record the SSL evaluation protocol before using test results.')

for cache, checkpoint in [(CACHE_R3D_KINETICS, None),
                          (CACHE_R3D_UCF_SSL, str(full_ssl_dir / 'best.pt'))]:
    make_r3d_cache(DATA / 'test.jsonl', cache, checkpoint)

def evaluate_r3d_arm(cache, arm, seed):
    checkpoint = CODE / 'runs' / f'{arm}_s{seed}' / 'best.pt'
    output = CODE / 'runs' / f'{arm}_s{seed}_test.jsonl'
    if output.exists(): raise FileExistsError(f'Test output already exists: {output}')
    args = argparse.Namespace(train=None, val=str(DATA / 'test.jsonl'), cache=str(cache),
        vocab=str(DATA / 'vocab.json'), output=str(output), steps=0, batch_size=64,
        hidden=256, lr=0.0003, seed=seed, question_only=False, init=None, resume=None,
        eval_checkpoint=str(checkpoint), mode='full')
    train_main(args)

for seed in [42, 43, 44]:
    evaluate_r3d_arm(CACHE_R3D_KINETICS, 'r3d_kinetics', seed)
    evaluate_r3d_arm(CACHE_R3D_UCF_SSL, 'r3d_ucf_ssl', seed)